# Model selection (AIC, BIC, MDL)

Information criteria reward likelihood but charge for model complexity.

AIC, BIC, and MDL prevent likelihood alone from rewarding every extra parameter. The notebook computes the criteria once and then measures the held-out log loss of selected classifiers across the ladder. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    cancer = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", cancer.data, cancer.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    model = LogisticRegression(max_iter=2000)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def split_scaled(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=3, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def fit_logistic(x_tr, y_tr, C=1.0):
    model = LogisticRegression(C=C, max_iter=2500)
    model.fit(x_tr, y_tr)
    return model


def classification_error(y_true, y_pred):
    return float(1.0 - accuracy_score(y_true, y_pred))


def lesson_score(losses, cost, alternative):
    raw = round(float(np.mean(losses)), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    relative_gap = gap / alternative
    stabilized = 0.8 * score
    return {
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stabilized": stabilized,
    }


def preview_ladder(rungs):
    for name, X, y in rungs:
        labels, counts = np.unique(y, return_counts=True)
        info = dict(zip(labels.tolist(), counts.tolist()))
        print(f"{name}: X={X.shape}, classes={info}, sample={np.round(X[:2], 3).tolist()}")


## The concept, built once on D1

The lesson formula is $$AIC=2k-2\ln L,\qquad BIC=k\ln n-2\ln L$$. We first rebuild the exact lesson arithmetic before using a reusable method on larger data.

In [ ]:

def model_selection_aic_bic_mdl_method(losses, cost, alternative):
    """Recompute the lesson raw average, cost-aware score, and alternative gap."""
    losses = np.asarray(losses, dtype=float)
    summary = lesson_score(losses, cost, alternative)
    return summary


losses = np.array([0.235, 0.096, 0.522], dtype=float)
summary = model_selection_aic_bic_mdl_method(losses, cost=0.100, alternative=0.424)

assert abs(summary["raw"] - 0.284) < 1e-12
assert abs(summary["score"] - 0.384) < 1e-12
assert abs(summary["gap"] - 0.040) < 1e-12
assert abs(summary["relative_gap"] - 0.094) < 0.001
assert abs(summary["stabilized"] - 0.307) < 0.001

print("losses:", losses.tolist())
print("raw average:", round(summary["raw"], 3))
print("score with cost:", round(summary["score"], 3))
print("gap to alternative:", round(summary["gap"], 3))
print("relative gap:", round(summary["relative_gap"], 3))


The same score object also carries the stabilized launch number and, for this topic, the topic-specific metric calculation used on the toy D1 counts or residuals.

In [ ]:

def information_criteria(log_likelihood, k, n):
    aic = 2.0 * k - 2.0 * log_likelihood
    bic = k * math.log(n) - 2.0 * log_likelihood
    mdl = 0.5 * bic
    return aic, bic, mdl

aic, bic, mdl = information_criteria(log_likelihood=-12.0, k=3, n=30)
assert round(aic, 3) == 30.0
assert round(bic, 3) == 34.204


print("toy AIC:", round(aic, 3))
print("toy BIC:", round(bic, 3))
print("toy MDL proxy:", round(mdl, 3))


## The dataset ladder

The notebook embeds the shared `clf_ladder()` source so it is self-contained in Colab. D1 is inspectable; D5 is real, higher-dimensional, and imbalanced enough to make the checks matter.

In [ ]:

rungs = clf_ladder()
preview_ladder(rungs)


## Run AIC/BIC-style selection across D1–D5

For each rung we fit several logistic models, select by BIC on the training split, and track held-out negative log loss.

In [ ]:

def safe_log_loss(y_true, probs, classes):
    return float(log_loss(y_true, probs, labels=classes))


def criterion_rung(X, y):
    x_tr, x_te, y_tr, y_te = split_scaled(X, y)
    classes = np.unique(y)
    candidates = []
    for C in [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
        model = fit_logistic(x_tr, y_tr, C=C)
        train_probs = model.predict_proba(x_tr)
        heldout_probs = model.predict_proba(x_te)
        train_nll = safe_log_loss(y_tr, train_probs, classes) * len(y_tr)
        heldout_nll = safe_log_loss(y_te, heldout_probs, classes)
        k = model.coef_.size + model.intercept_.size
        aic = 2.0 * k + 2.0 * train_nll
        bic = k * math.log(len(y_tr)) + 2.0 * train_nll
        mdl = 0.5 * bic
        candidates.append({"C": C, "aic": aic, "bic": bic, "mdl": mdl, "heldout_nll": heldout_nll})
    chosen = min(candidates, key=lambda row: row["bic"])
    return candidates, chosen

results = []
for name, X, y in rungs:
    candidates, chosen = criterion_rung(X, y)
    results.append({"name": name, "candidates": candidates, "chosen": chosen, "metric": chosen["heldout_nll"]})
    print(f"{name}: chosen_C={chosen['C']}, heldout_log_loss={chosen['heldout_nll']:.3f}, BIC={chosen['bic']:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, row in zip(axes[0], results):
    Cs = [item["C"] for item in row["candidates"]]
    bics = [item["bic"] for item in row["candidates"]]
    ax.bar([str(C) for C in Cs], bics, color="seagreen")
    ax.set_title(row["name"].split(" (")[0], fontsize=8)
    ax.tick_params(axis="x", labelrotation=45)
axes[1, 0].plot(range(1, 6), [row["metric"] for row in results], marker="o")
axes[1, 0].set_title("held-out log loss vs rung")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("negative log loss")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

Maximum likelihood alone prefers flexibility too easily. AIC/BIC/MDL add the missing cost before ranking candidates.

In [ ]:

name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = split_scaled(X, y)
rows = []
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    model = fit_logistic(x_tr, y_tr, C=C)
    preds = model.predict(x_te)
    raw_loss = classification_error(y_te, preds)
    complexity_cost = 0.100 * math.log10(C + 1.0) / math.log10(11.0)
    decision_score = raw_loss + complexity_cost
    rows.append((C, raw_loss, complexity_cost, decision_score))

raw_best = min(rows, key=lambda row: row[1])
fixed_best = min(rows, key=lambda row: row[3])
print("D5 candidate table: C, raw validation loss, cost, decision score")
for row in rows:
    print(tuple(round(value, 4) for value in row))
print("wrong raw-only choice:", raw_best)
print("cost-aware choice:", fixed_best)
print("decision-score improvement:", round(raw_best[3] - fixed_best[3], 4))
assert fixed_best[3] <= raw_best[3] + 1e-12


## Evaluate it + Practice

- Track `held-out log loss` against a no-skill baseline before trusting the result.
- Sanity-check that D1 reproduces the lesson numbers exactly and that D5 is not a toy shortcut.
- Ablation: rank candidates by training likelihood only; the metric should get worse or the selected model should become less stable.
- Failure signals: large validation gap, unstable threshold/criterion, or a cost-aware score that disagrees with the raw metric.
- Report both the raw metric and the cost-aware decision score when the lesson formula includes a cost.

Practice prompts:
1. Change one candidate hyperparameter and recompute the D5 table.
2. Replace the no-skill baseline and explain whether `held-out log loss` moved for the right reason.
3. Add one diagnostic plot that would catch the named pitfall for model selection.